# Internship Skill Gap Finder

My app helps students targeting analyst internships decide which portfolio project to build next by comparing job tasks across target roles with the skills they have practiced.

**Try it:** Upload both CSVs from the repo, run all cells, then change the roles or practiced skills in Step 3. The sample selections are for demonstration.

## 1. Load the data
This CSV is an extract of all 48 published task statements for the three selected occupations in the O*NET 30.3 Task Statements dataset. The task wording and IDs come directly from the source.

In [3]:
import json
from urllib.error import URLError
from urllib.parse import urlencode
from urllib.request import urlopen

import pandas as pd
from IPython.display import display

tasks = pd.read_csv("role_tasks.csv")
print(f"Loaded {len(tasks)} tasks across {tasks['role'].nunique()} roles.")

Loaded 48 tasks across 3 roles.


## 2. Match task text to ESCO skills
The rules below recognize phrases in the original task text and translate them into search terms. The notebook sends each search term to the European Commission's ESCO API and reads the first English skill result. Tasks outside these rules stay blank in the skill column. It saves the official skill label as a new column called `esco_skill`. One lookup is made for each unique phrase. The included snapshot provides a way to run the analysis if the API is unavailable.

Check the first match beside each task. ESCO ranks search results, and a high-ranked match still needs a relevance check.

In [5]:
API = "https://ec.europa.eu/esco/api/search"
USE_LIVE_API = True

def find_esco_skill(phrase):
    parameters = {"text": phrase, "type": "skill", "language": "en", "limit": 1}
    with urlopen(f"{API}?{urlencode(parameters)}", timeout=8) as response:
        data = json.load(response)
    results = data.get("_embedded", {}).get("results", [])
    if not results:
        raise ValueError(f"No ESCO result for {phrase!r}")
    return results[0]["title"], results[0]["uri"]

# Each rule looks for a text fragment in the original O*NET task statement.
SEARCH_TERMS = {
    "analyze financial or operational performance": "analyse financial performance",
    "analyze financial data, such as income growth": "analyse financial performance",
    "future investment-risk trends": "analyse financial risk",
    "analyze credit data and financial statements": "analyse financial risk",
    "present oral or written reports": "present reports",
    "document findings of study": "present reports",
    "prepare reports that include the degree of risk": "create financial reports",
    "analyze data gathered and develop solutions": "perform business analysis",
    "interview personnel and conduct on-site observation": "analyse business processes",
}
tasks["phrase"] = pd.NA
for fragment, search_term in SEARCH_TERMS.items():
    matches_text = tasks["task_text"].str.lower().str.contains(fragment, regex=False)
    tasks.loc[matches_text, "phrase"] = search_term
phrases = tasks["phrase"].dropna().unique()
try:
    if USE_LIVE_API:
        matches = []
        for phrase in phrases:
            skill, uri = find_esco_skill(phrase)
            matches.append((phrase, skill, uri))
        lookup = pd.DataFrame(matches, columns=["phrase", "esco_skill", "esco_uri"])
        source = "Live ESCO search"
    else:
        lookup = pd.read_csv("esco_snapshot.csv")
        source = "Saved ESCO snapshot"
except (URLError, TimeoutError, ValueError) as error:
    lookup = pd.read_csv("esco_snapshot.csv")
    source = f"Saved ESCO snapshot after live lookup failed ({type(error).__name__})"

tasks = tasks.merge(lookup, on="phrase", how="left", validate="many_to_one")
print(source)
display(tasks[["phrase", "esco_skill", "esco_uri"]].dropna().drop_duplicates())

                           phrase                                  esco_skill                                                               esco_uri
2       perform business analysis                   perform business analysis  http://data.europa.eu/esco/skill/27ed854c-15b8-4ba2-90e9-ae888a219703
3                 present reports                             present reports  http://data.europa.eu/esco/skill/d0f2f8a7-d935-4a6d-8e54-dcc3e5e2cbeb
5      analyse business processes                  analyse business processes  http://data.europa.eu/esco/skill/56042303-972b-41be-9310-58467599ff7e
11         analyse financial risk                      analyse financial risk  http://data.europa.eu/esco/skill/eb8babfb-ab91-4937-a032-6a79f82e9b7f
14       create financial reports                   create a financial report  http://data.europa.eu/esco/skill/dd7c9dfd-6e2c-46fd-aefc-369402d0200c
15  analyse financial performance  analyse financial performance of a company  http://data.europa.eu/esco/

Live ESCO search


## 3. Pick your target roles and skills
Copy role names and skill labels from the notebook. Remove the sample practiced skills to start with an empty checklist. Run this cell and all cells below it every time you change your choices.

In [7]:
TARGET_ROLES = ["Investment analysis", "Credit analysis", "Consulting"]
PRACTICED_SKILLS = ["present reports"]  # Example selection

# Each portfolio project lists the skills its deliverables practice.
PROJECTS = {
    "Company and credit memo": [
        "analyse financial performance of a company", "analyse financial risk", "create a financial report"
    ],
    "Small business process audit": [
        "perform business analysis", "analyse business processes", "present reports"
    ],
    "Investor presentation": [
        "analyse financial performance of a company", "present reports"
    ],
}

if not TARGET_ROLES or not set(TARGET_ROLES).issubset(tasks["role"]):
    raise ValueError("Copy at least one role name exactly as it appears in the data.")
if not set(PRACTICED_SKILLS).issubset(tasks["esco_skill"]):
    raise ValueError("Copy practiced skill labels exactly as they appear in Step 2.")

## 4. Define the four answers
The class stores the selected roles' tasks and the user's checklist. Each method answers one question. We count distinct roles for each skill so a role with repeated tasks cannot inflate the score.

In [9]:
class SkillGapFinder:
    def __init__(self, tasks, roles, practiced):
        self.tasks = tasks[tasks["role"].isin(roles)].copy()
        self.practiced = practiced

    def role_requirements(self):
        columns = ["role", "task_text", "esco_skill"]
        coverage = self.tasks.groupby("role")["esco_skill"].agg(
            published_tasks="size", matched_tasks="count"
        )
        return coverage, self.tasks[columns].dropna().sort_values("role")

    def skill_overlap(self):
        role_counts = self.tasks.groupby("esco_skill")["role"].nunique()
        return role_counts.sort_values(ascending=False).rename("roles").to_frame()

    def skill_gaps(self, overlap):
        missing = ~overlap.index.isin(self.practiced)
        return overlap[missing]

    def recommend_projects(self, gaps):
        rows = []
        for project, skills in PROJECTS.items():
            covered = gaps[gaps.index.isin(skills)]
            rows.append({
                "project": project,
                "score": int(covered["roles"].sum()),
                "gaps_addressed": ", ".join(covered.index),
            })
        scores = pd.DataFrame(rows)
        return scores.sort_values(["score", "project"], ascending=[False, True])

app = SkillGapFinder(tasks, TARGET_ROLES, PRACTICED_SKILLS)

## Q1. What do people in my target roles do?

Compare how many tasks O*NET lists for each occupation and how many the ESCO lookup covered. Then read the source statements that received a skill label.

In [11]:
coverage, matched_tasks = app.role_requirements()
display(coverage)
display(matched_tasks)

                     published_tasks  matched_tasks
role                                               
Consulting                        11              3
Credit analysis                   11              3
Investment analysis               26              3

                   role                                                                                                                                        task_text                                  esco_skill
2            Consulting                                                                Analyze data gathered and develop solutions or alternative methods of proceeding.                   perform business analysis
3            Consulting                 Document findings of study and prepare recommendations for implementation of new systems, procedures, or organizational changes.                             present reports
5            Consulting     Interview personnel and conduct on-site observation to ascertain unit functions, work performed, and methods, equipment, and personnel used.                  analyse business processes
11      Credit analysis                      Analyze credit data and financial statements to determine the degree of risk involved in extending cred

## Q2. Which skills show up across my target roles?

Group by skill and count distinct roles. A count of 2 means the skill appears in two selected roles.

In [13]:
overlap = app.skill_overlap()
display(overlap)

                                            roles
esco_skill                                       
analyse financial performance of a company      2
analyse financial risk                          2
present reports                                 2
analyse business processes                      1
create a financial report                       1
perform business analysis                       1

## Q3. Which skills should I practice?

Keep the skills the user has yet to check off. The `~` flips True to False and False to True.

In [15]:
gaps = app.skill_gaps(overlap)
display(gaps)

                                            roles
esco_skill                                       
analyse financial performance of a company      2
analyse financial risk                          2
analyse business processes                      1
create a financial report                       1
perform business analysis                       1

## Q4. What portfolio project should I build next?
For each project, look up the skills it would practice. Add one point for each selected role where each missing skill appears. If two projects have the same score, list them alphabetically. Use the scores to choose where to spend your next study session.

In [17]:
scores = app.recommend_projects(gaps)
display(scores)
best = scores.iloc[0]
if best["score"] == 0:
    print("No listed project addresses a remaining gap for these choices.")
else:
    print(f"Build next: {best['project']} (score {best['score']}).")
    if best["project"] == "Company and credit memo":
        print("Use a public company filing. Compare the last two years of results,")
        print("explain a downside scenario, and summarize the credit risks in one page.")
    elif best["project"] == "Small business process audit":
        print("Map one process, identify a bottleneck, and present a measurable improvement.")
    else:
        print("Summarize a company's performance and present the takeaways in five slides.")

                        project  score                                                                                 gaps_addressed
0       Company and credit memo      5  analyse financial performance of a company, analyse financial risk, create a financial report
2         Investor presentation      2                                                     analyse financial performance of a company
1  Small business process audit      2                                          analyse business processes, perform business analysis

Build next: Company and credit memo (score 5).
Use a public company filing. Compare the last two years of results,
explain a downside scenario, and summarize the credit risks in one page.


## Limitations
The dataset contains 48 original O*NET records. The skill rules cover nine of those statements, three per occupation. Skill rankings describe that matched subset. They cannot establish how often employers request a skill or whether an applicant is qualified. One API label is assigned per task, and the projects cover skills by design. Use the output to plan practice, then check the responsibilities in actual postings you care about.